In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
BASE_DIR = "./cesnet-institutions-throughput/institutions/agg_1_hour/" 

In [3]:
def introduce_missing_data(df, missing_rate, seed=42):
    rng = np.random.default_rng(seed)
    df_missing = df.copy()
    mask = rng.random(len(df_missing)) < missing_rate
    df_missing.loc[mask, "throughput_bps"] = np.nan
    print(f"Introduced {missing_rate * 100}% missing data.")
    # print(f"Mask: {mask}")
    return df_missing

datasets_missing = {}

for file in os.listdir(BASE_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(BASE_DIR, file))
        base_key = file.removesuffix(".throughput.csv")  
        datasets_missing[base_key] = {} 
        
        for rate in [0.1, 0.2, 0.3, 0.4]:
            df_missing = introduce_missing_data(df, missing_rate=rate, seed=42)
            rate_key = f"{int(rate * 100)}"
            datasets_missing[base_key][rate_key] = df_missing

datasets_missing  # DataFrame com 10% de dados faltantes

Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduc

{'232': {'10':                      time  throughput_bps
  0     2023-10-09 00:00:00    1.012115e+06
  1     2023-10-09 01:00:00    6.334464e+05
  2     2023-10-09 02:00:00    1.349004e+06
  3     2023-10-09 03:00:00    1.365274e+06
  4     2023-10-09 04:00:00             NaN
  ...                   ...             ...
  6712  2024-07-14 17:00:00    3.340266e+05
  6713  2024-07-14 18:00:00    6.502300e+05
  6714  2024-07-14 19:00:00    6.546512e+05
  6715  2024-07-14 20:00:00    8.950076e+04
  6716  2024-07-14 21:00:00    6.447292e+04
  
  [6717 rows x 2 columns],
  '20':                      time  throughput_bps
  0     2023-10-09 00:00:00    1.012115e+06
  1     2023-10-09 01:00:00    6.334464e+05
  2     2023-10-09 02:00:00    1.349004e+06
  3     2023-10-09 03:00:00    1.365274e+06
  4     2023-10-09 04:00:00             NaN
  ...                   ...             ...
  6712  2024-07-14 17:00:00    3.340266e+05
  6713  2024-07-14 18:00:00    6.502300e+05
  6714  2024-07-14 19:00:00

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [5]:
from sklearn.impute import KNNImputer


In [6]:
def impute_knn_imputer(df_missing, k=5):
    df_imp = df_missing.copy()

    imputer = KNNImputer(n_neighbors=k, weights="uniform")
    imputed_values = imputer.fit_transform(df_imp[["throughput_bps"]])

    df_imp["throughput_bps"] = imputed_values[:, 0]
    return df_imp

In [7]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.structural import UnobservedComponents

In [8]:
def impute_kalman(
    df_missing,
    model="arima",
    # --- ARIMA/SARIMA params ---
    arima_order=(1, 1, 1),
    seasonal_order=(0, 0, 0, 0),
    # --- Estrutural params ---
    level="local level",         # opções comuns: "local level", "local linear trend"
    seasonal_period=None         # por ex., 24 p/ sazonalidade diária em dados horários
):
    """
    Imputa NaNs em 'throughput_bps' via Kalman smoothing.

    Parâmetros
    ----------
    df_missing : pd.DataFrame
        DataFrame com colunas 'time' (datetime string com timezone) e 'throughput_bps'.
    model : {'arima', 'structural'}
        Escolhe a abordagem:
        - 'arima': ajusta SARIMAX(ARIMA/SARIMA) e usa as previsões in-sample (Kalman).
        - 'structural': ajusta UnobservedComponents (nível/trend/seasonal) + Kalman.
    arima_order : tuple
        Ordem (p, d, q) do ARIMA.
    seasonal_order : tuple
        Ordem sazonal (P, D, Q, s) para SARIMA.
    level : str
        Componente de nível do modelo estrutural (ex.: 'local level', 'local linear trend').
    seasonal_period : int or None
        Período sazonal para o modelo estrutural (ex.: 24, 7*24, etc.). None = sem sazonalidade.

    Retorna
    -------
    df_imputed : pd.DataFrame
        Cópia de df_missing com 'throughput_bps' imputado nos pontos NaN.
    """
    df_imp = df_missing.copy()

    # Garante dtype datetime (assumido correto e com tz, sem tratamento de erros)
    t = pd.to_datetime(df_imp["time"])
    y = df_imp["throughput_bps"].astype(float)

    # Máscara de faltantes
    miss_mask = y.isna()

    if model == "arima":
        # SARIMAX lida com NaNs na endógena e usa Kalman internamente
        mod = SARIMAX(
            y,
            order=arima_order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        # Previsão in-sample (predicted_mean) já incorpora filtro/smoother de Kalman
        y_hat = res.get_prediction().predicted_mean

    elif model == "structural":
        # Modelo estrutural: nível / tendência / sazonal (state-space) + Kalman
        # Ex.: level='local level' ou 'local linear trend'
        #     seasonal_period define sazonalidade (ex.: 24 p/ hora, 7*24 p/ semanal, etc.)
        ucm = UnobservedComponents(
            y,
            level=level,
            seasonal=seasonal_period  # None => sem sazonal
        )
        res = ucm.fit(disp=False)

        # predicted_mean é a série observável estimada (alisada) pelo modelo
        y_hat = res.get_prediction().predicted_mean

    else:
        raise ValueError("model must be 'arima' or 'structural'")

    # Imputa apenas onde havia NaN, preservando os valores observados
    y_imp = y.copy()
    y_imp[miss_mask] = y_hat[miss_mask].values

    df_imp["throughput_bps"] = y_imp.values
    return df_imp


In [9]:
# --- Moving Average (SMA) centrada ---
def impute_moving_average(df_missing, window=5, center=True):
    """
    Imputa NaNs em 'throughput_bps' usando média móvel (SMA).
    
    Ideia:
      - Calcula a média dos vizinhos dentro de uma janela.
      - Aqui usamos janela centrada (anterior e posterior) e min_periods=1
        para sempre produzir valor mesmo com poucas observações na borda.
      - Substitui apenas os pontos que eram NaN originalmente.

    Parâmetros
      window: tamanho da janela (3–10 usualmente)
      center: True para janela centrada (vizinho passado e futuro)
    """
    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"]

    # Média móvel centrada; ignora NaNs automaticamente; min_periods=1 para não “perder” bordas
    sma = y.rolling(window=window, center=center, min_periods=1).mean()

    mask_missing = y.isna()
    df_imp.loc[mask_missing, "throughput_bps"] = sma[mask_missing].values
    return df_imp


# --- Exponentially Weighted Moving Average (EWMA) ---
def impute_ewma(df_missing, alpha=0.2):
    """
    Imputa NaNs em 'throughput_bps' usando EWMA (média móvel exponencial).
    
    Ideia:
      - EWMA pondera mais os valores recentes (decaimento controlado por alpha).
      - Aqui usamos a forma “causal” (somente passado) com adjust=False (recursivo padrão).
      - Substitui apenas os pontos que eram NaN originalmente.

    Parâmetros
      alpha: 0.1–0.3 é um intervalo comum; quanto maior, mais peso ao mais recente.
    """
    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"]

    ew = y.ewm(alpha=alpha, adjust=False).mean()

    mask_missing = y.isna()
    df_imp.loc[mask_missing, "throughput_bps"] = ew[mask_missing].values
    return df_imp


In [10]:
# ----------------------------
# PERIOD (n_lines) ESTIMATION
# ----------------------------

def _acf(y, max_lag): 
    """Biased ACF up to max_lag (lag 0..max_lag). NaNs are linearly interpolated first."""
    y = pd.Series(y).interpolate(limit_direction="both").to_numpy()
    y = y - np.nanmean(y)
    n = len(y)
    acf_vals = np.empty(max_lag + 1)
    denom = np.dot(y, y) + 1e-12
    for lag in range(max_lag + 1):
        acf_vals[lag] = np.dot(y[: n - lag], y[lag:]) / denom
    return acf_vals

def estimate_period(
    y,
    min_period = 4,
    max_period = None
) -> int:
    """
    Pick period (n_lines) automatically using ACF peak 
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    if max_period is None:
        max_period = max(7, min(n // 4, 1000))  # sensible cap

    if n < min_period * 2:
        # too short — just return something minimum
        return max(min_period, min(n, 24))

    acf_vals = _acf(y, max_period)
    # Ignore lag 0; pick the best lag in [min_period, max_period]
    candidate_lags = np.arange(min_period, max_period + 1)
    best_lag = candidate_lags[np.argmax(acf_vals[min_period: max_period + 1])]

    return int(best_lag)

# ---------------------------------
# FOLDING (TEMPORAL MATRIX)
# ---------------------------------

def fold_series_to_matrix(y, period):
    """
    Fold 1D series into a (n_blocks, period) matrix. 
    Pads the last block with NaN if needed.
    Returns (M, original_len).
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    n_blocks = int(np.ceil(n / period))
    pad_len = n_blocks * period - n
    if pad_len > 0:
        y = np.concatenate([y, np.full(pad_len, np.nan)])
    M = y.reshape(n_blocks, period)
    return M, n

def unfold_matrix_to_series(M, original_len): 
    """Inverse of fold: row-wise flatten and trim to original length."""
    y = M.reshape(-1)
    return y[:original_len]

# ----------------------------
# SVD + RANK SELECTION
# ----------------------------

def svd_rank(M_filled, energy = 0.9):
    """
    Compute SVD and choose rank r by cumulative explained 'energy' (sum of singular values).
    """
    U, s, Vt = np.linalg.svd(M_filled, full_matrices=False)
    s2 = s**2
    cum = np.cumsum(s2) / (np.sum(s2) + 1e-24)
    r = int(np.searchsorted(cum, energy) + 1)
    r = max(1, min(r, min(M_filled.shape)))
    return U, s, Vt, r

# -------------------------------------------
# KNN IMPUTE USING ROW EMBEDDINGS FROM SVD
# -------------------------------------------

def _warm_start_fill(M: np.ndarray) -> np.ndarray: # coloca mediana em tudo e vapo só pra começar
    """Column-wise median fill as a stable warm start."""
    M_filled = M.copy()
    col_medians = np.nanmedian(M_filled, axis=0)
    inds = np.where(np.isnan(M_filled))
    M_filled[inds] = np.take(col_medians, inds[1])
    return M_filled

def impute_with_knn_in_latent( 
    M: np.ndarray,
    k: int = 5,
    energy: float = 0.9,
    allow_future: bool = True
) -> np.ndarray:
    """
    Impute NaNs in M by KNN in SVD latent space (rows ≈ cycles/weeks). que porra eh latent space
    For each missing cell (i,j), find k nearest rows to row i in latent space
    among those with M[row, j] observed (and optionally row < i).
    """
    M_filled0 = _warm_start_fill(M)
    U, s, Vt, r = svd_rank(M_filled0, energy=energy)
    # Row embeddings (T x r): U_r * S_r
    Z = U[:, :r] * s[:r]  # broadcasting: each column of U scaled by s

    M_imp = M.copy()
    T, P = M.shape
    eps = 1e-8

    # Precompute which rows have each column observed
    observed_mask = ~np.isnan(M)
    for i in range(T):
        # indices (columns) that are missing in row i
        miss_cols = np.where(~observed_mask[i])[0]
        if len(miss_cols) == 0:
            continue
        zi = Z[i]

        # Candidate rows for neighbors (global, filtered below per col)
        if allow_future:
            candidate_rows_global = np.arange(T)
        else:
            candidate_rows_global = np.arange(0, i)  # only past

        if len(candidate_rows_global) == 0:
            # If we can't use past (i==0), allow future just for this row:
            candidate_rows_global = np.arange(T)

        # Distances in latent space to all candidates
        Zc = Z[candidate_rows_global]
        dists = np.linalg.norm(Zc - zi[None, :], axis=1)
        dists = dists + eps  # avoid zero

        for j in miss_cols:
            # keep only candidates that have this column observed
            obs_rows = candidate_rows_global[observed_mask[candidate_rows_global, j]]
            if len(obs_rows) == 0:
                # fall back to warm-start value if nothing observed
                M_imp[i, j] = M_filled0[i, j]
                continue

            # distances for those rows
            d = np.linalg.norm(Z[obs_rows] - zi[None, :], axis=1) + eps
            # k nearest
            if len(d) > k:
                idx = np.argpartition(d, k)[:k]
                nn_rows = obs_rows[idx]
                d = d[idx]
            else:
                nn_rows = obs_rows

            w = 1.0 / d  # inverse-distance weights
            vals = M[nn_rows, j]
            # safety: if still NaN (shouldn't happen), drop them
            ok = ~np.isnan(vals)
            if not np.any(ok):
                M_imp[i, j] = M_filled0[i, j]
            else:
                vals = vals[ok]
                w = w[ok]
                M_imp[i, j] = np.sum(w * vals) / np.sum(w)

    return M_imp

# ----------------------------
# IMPUTATION FUNCTION
# ----------------------------

def impute_throughput_svd_knn(
    df: pd.DataFrame,
    col: str = "throughput_bps",
    min_period: int = 4,
    max_period = None,
    energy: float = 0.9,
    k: int = 5,
    allow_future: bool = True
):
    """
    Full pipeline:
    1) auto period detection -> n_lines (period)
    2) fold into (n_blocks, period) matrix
    3) SVD -> latent row embeddings
    4) KNN in latent space to impute NaNs
    5) unfold back to 1D series
    Returns (imputed_series, diagnostics).
    """
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in df.")

    y = df[col].to_numpy(dtype=float)
    original_index = df.index

    # Detect period (n_lines)
    period = estimate_period(y, min_period=min_period, max_period=max_period)

    # Fold
    M, orig_len = fold_series_to_matrix(y, period=period)

    # Impute in latent KNN space
    M_imp = impute_with_knn_in_latent(M, k=k, energy=energy, allow_future=allow_future)

    # Unfold
    y_imp = unfold_matrix_to_series(M_imp, original_len=orig_len)

    print(y_imp)

    diagnostics = {
        "period_estimated": period,
        "matrix_shape": M.shape,
        "rank_energy_target": energy,
        "k_neighbors": k,
        "allow_future": allow_future,
    }
    return pd.Series(y_imp, index=original_index, name=f"{col}_imputed"), diagnostics



In [11]:
results = []

def impute_linear_interpolation(df_missing):
    df_imputed = df_missing.copy()
    df_imputed["throughput_bps"] = df_imputed["throughput_bps"].interpolate(
        method="linear", limit_direction="both"
    )
    return df_imputed

# def evaluate_imputation(mask_missing, df, df_imputed, method):
#     # real and imputed values where theres missing
#     y_true = df.loc[mask_missing, "throughput_bps"].values
#     y_pred = df_imputed.loc[mask_missing, "throughput_bps"].values
    
#     if len(y_true) > 0: 
#         rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#         nrmse = rmse / (y_true.max() - y_true.min()) # range
#         nrmse_mean = rmse / y_true.mean() # mean
#         mae = mean_absolute_error(y_true, y_pred)
#         mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
#         r2 = r2_score(y_true, y_pred)
        
#         results.append({
#             "file": file,
#             "rate": rate,
#             "method": method,
#             "rmse": rmse,
#             "nrmse": nrmse, # range
#             "nrmse_mean": nrmse_mean, # mean
#             "mae": mae,
#             "mape": mape,
#             "r2": r2,
#         })

#         return results

# for file, rates_dict in datasets_missing.items():
#     for rate, df_missing in rates_dict.items():
#         mask_missing = df_missing["throughput_bps"].isna()
        
#         # linear interpolation
#         df_imputed = impute_linear_interpolation(df_missing)
#         results = evaluate_imputation(mask_missing, df, df_imputed, "linear interpolation")

#         df_imputed_knn = impute_knn_imputer(df_missing, k=5)
#         results = evaluate_imputation(mask_missing, df, df_imputed_knn, "knn imputer (k=5)")

#         # 4) Kalman - ARIMA (ex.: ARIMA(1,1,1); sem sazonalidade)
#         df_kalman_arima = impute_kalman(
#             df_missing,
#             model="arima",
#             arima_order=(1, 1, 1),
#             seasonal_order=(0, 0, 0, 0)
#         )
#         results = evaluate_imputation(mask_missing, df, df_kalman_arima, "kalman arima (1,1,1)")

#         # 5) Kalman - Modelo Estrutural (nível local + sazonal diária de 24, se fizer sentido)
#         df_kalman_struct = impute_kalman(
#             df_missing,
#             model="structural",
#             level="local level",
#             seasonal_period=None  # ou 24, 7*24 etc., conforme seu dado
#         )

#         # df_sma = impute_moving_average(df_missing, window=5, center=True)
#         # results = evaluate_imputation(mask_missing, df, df_sma, "moving average (win=5, centered)")

#         # EWMA (alpha 0.2)
#         df_ewma = impute_ewma(df_missing, alpha=0.2)
#         results = evaluate_imputation(mask_missing, df, df_ewma, "ewma (alpha=0.2)")

#         results = evaluate_imputation(mask_missing, df, df_kalman_struct, "kalman structural (level)")
        
#         df_hankel = impute_hankel_knn_svd(df_missing, window=72, k=5, blend=0.2, seasonal_period=None)
#         results = evaluate_imputation(mask_missing, df, df_hankel, "hankel+knn+svd (L=72, k=5, λ=0.2)")

#         df_imputed, diag = impute_throughput_svd_knn(
#                 df_missing,
#                 col="throughput_bps",
#                 min_period=24,   # adjust for your sampling rate
#                 max_period=1000, # search window for period detection
#                 energy=0.9,
#                 k=5,
#                 allow_future=True
#             )

#         # Merge back into DataFrame with same structure
#         df_result = df_missing.copy()
#         df_result["throughput_bps"] = df_imputed.values

#         # --------------------------
#         # Evaluate
#         # --------------------------
#         results = evaluate_imputation(
#             mask_missing, 
#             df, df_result, 
#             method="Hankel+SVD+KNN"
#         )
        
# df_results = pd.DataFrame(results)
# df_results.head()
# df_results.to_csv("results.csv", index=False)

# print("Resultados salvos em results.csv")


In [12]:
# def create_temporal_matrix(df, n_lines=100):
#     throughput_vector = df["throughput_bps"].values
#     columns_quantity = df.shape[1]
#     temporal_matrix = (
#         throughput_vector[: columns_quantity * n_lines]
#         .reshape(columns_quantity, n_lines)
#         .T
#     )
#     return temporal_matrix


In [13]:
# --- Mean (média) ---
def impute_mean(df_missing):
    """
    Imputa NaNs em 'throughput_bps' usando a média global dos valores observados.
    Substitui apenas os pontos originalmente NaN.
    """
    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"]
    mu = y.mean(skipna=True)
    mask_missing = y.isna()
    df_imp.loc[mask_missing, "throughput_bps"] = mu
    return df_imp


# --- Simple Moving Average (média móvel simples - SMA) ---
def impute_moving_average(df_missing, window=5, center=False):
    """
    Imputa NaNs em 'throughput_bps' usando média móvel simples (rolling mean).
    
    Parâmetros:
      window: tamanho da janela (ex.: 3, 5, 7...)
      center: se True, janela centralizada (melhor p/ off-line); se False, causal.
    """
    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"]
    sma = y.rolling(window=window, min_periods=1, center=center).mean()
    mask_missing = y.isna()
    df_imp.loc[mask_missing, "throughput_bps"] = sma[mask_missing].values
    return df_imp


# --- Iterative SVD (fancyimpute) ---
def impute_softimpute(df_missing, max_iters=200, convergence_threshold=1e-5, random_state=42, shrinkage_value=None):
    """
    Imputa NaNs em 'throughput_bps' usando SoftImpute (SVD com thresholding),
    preservando valores observados e preenchendo só os NaNs.
    """
    from fancyimpute import SoftImpute

    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"].astype(float).to_numpy()
    mask_missing = np.isnan(y)

    Y = y.reshape(-1, 1)
    imputer = SoftImpute(max_iters=max_iters,
                         convergence_threshold=convergence_threshold,
                         shrinkage_value=shrinkage_value,
                         verbose=False)
    Y_hat = imputer.fit_transform(Y).reshape(-1)

    y_out = y.copy()
    y_out[mask_missing] = Y_hat[mask_missing]
    df_imp["throughput_bps"] = y_out
    return df_imp

# --- Forward fill (preencher com último valor conhecido) ---
def impute_ffill(df_missing):
    """
    Imputa NaNs em 'throughput_bps' com forward fill (ffill).
    Substitui apenas os pontos originalmente NaN.
    """
    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"]
    y_ffill = y.ffill()
    mask_missing = y.isna()
    df_imp.loc[mask_missing, "throughput_bps"] = y_ffill[mask_missing].values
    return df_imp


# --- Backward fill (preencher com próximo valor conhecido) ---
def impute_bfill(df_missing):
    """
    Imputa NaNs em 'throughput_bps' com backward fill (bfill).
    Substitui apenas os pontos originalmente NaN.
    """
    df_imp = df_missing.copy()
    y = df_imp["throughput_bps"]
    y_bfill = y.bfill()
    mask_missing = y.isna()
    df_imp.loc[mask_missing, "throughput_bps"] = y_bfill[mask_missing].values
    return df_imp


In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []

def evaluate_imputation(mask_missing, df_true, df_imp, method, file, rate):
    y_true = df_true.loc[mask_missing, "throughput_bps"].values
    y_pred = df_imp.loc[mask_missing, "throughput_bps"].values

    if y_true.size == 0:
        return results

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nrmse = rmse / (y_true.max() - y_true.min())
    nrmse_mean = rmse / y_true.mean()
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)

    results.append({
        "file": file,
        "rate": int(rate),
        "method": method,
        "rmse": rmse,
        "nrmse": nrmse,
        "nrmse_mean": nrmse_mean,
        "mae": mae,
        "mape": mape,
        "r2": r2,
    })
    return results

datasets_original = {}
datasets_missing = {}

for file in os.listdir(BASE_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(BASE_DIR, file))
        
        # salva o dataset original
        base_key = file.removesuffix(".throughput.csv")  
        datasets_original[base_key] = df.copy()  

        # gera versões com missing
        datasets_missing[base_key] = {}
        for rate in [0.1, 0.2, 0.3, 0.4]:
            df_missing = introduce_missing_data(df, missing_rate=rate, seed=42)
            rate_key = str(int(rate * 100))
            datasets_missing[base_key][rate_key] = df_missing

for file, rates_dict in datasets_missing.items():
    df = datasets_original[file].copy()

    for rate_key, df_missing in rates_dict.items():
        rate = int(rate_key)
        mask_missing = df_missing["throughput_bps"].isna()

        try:
            df_imp_series, diag = impute_throughput_svd_knn(
                df_missing,
                col="throughput_bps",
                min_period=24,
                max_period=1000,
            energy=0.9,
            k=10,
            allow_future=True
        )
            df_result = df_missing.copy()
            df_result["throughput_bps"] = df_imp_series.values
            evaluate_imputation(mask_missing, df, df_result, "Temporal_Matrix+SVD+KNN", file, rate)
        except Exception as e:
            print(f"Error occurred while imputing {file} at rate {rate}: {e}")
            continue
        
        df_lin = impute_linear_interpolation(df_missing)
        evaluate_imputation(mask_missing, df, df_lin, "linear interpolation", file, rate)

        df_knn = impute_knn_imputer(df_missing, k=5)
        evaluate_imputation(mask_missing, df, df_knn, "knn imputer (k=5)", file, rate)

        df_kalman_arima = impute_kalman(
            df_missing,
            model="arima",
            arima_order=(1, 1, 1),
            seasonal_order=(0, 0, 0, 0)
        )
        evaluate_imputation(mask_missing, df, df_kalman_arima, "kalman arima (1,1,1)", file, rate)

        df_kalman_struct = impute_kalman(
            df_missing,
            model="structural",
            level="local level",
            seasonal_period=None
        )
        evaluate_imputation(mask_missing, df, df_kalman_struct, "kalman structural (level)", file, rate)

        df_ewma = impute_ewma(df_missing, alpha=0.2)
        evaluate_imputation(mask_missing, df, df_ewma, "ewma (alpha=0.2)", file, rate)

        # df_hankel = impute_hankel_knn_svd(df_missing, window=72, k=5, blend=0.2, seasonal_period=None)
        # evaluate_imputation(mask_missing, df, df_hankel, "Hankel+KNN+SVD (L=72, k=5, λ=0.2)", file, rate)

        
        df_mean = impute_mean(df_missing)
        evaluate_imputation(mask_missing, df, df_mean, "mean", file, rate)

        # df_moving_average = impute_moving_average(df_missing)
        # evaluate_imputation(mask_missing, df, df_moving_average, "moving_average", file, rate)

        df_soft_impute = impute_softimpute(df_missing)
        evaluate_imputation(mask_missing, df, df_soft_impute, "soft_impute", file, rate)

        df_ffill = impute_ffill(df_missing)
        evaluate_imputation(mask_missing, df, df_ffill, "ffill", file, rate)

        
df_results = pd.DataFrame(results)
df_results.to_csv("results_total.csv", index=False)
print("Resultados salvos em results.csv")
df_results.head()


Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduced 10.0% missing data.
Introduced 20.0% missing data.
Introduced 30.0% missing data.
Introduced 40.0% missing data.
Introduc

/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[1012115.00222222  633446.35555556 1349003.78666667 ...  654651.22888889
   89500.76222222   64472.91777778]


/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[1012115.00222222  633446.35555556 1349003.78666667 ...  654651.22888889
   89500.76222222   64472.91777778]


...

[ 6364.9         4460.42       23011.30444444 ...  3966.48
 10713.55333333  1932.90888889]


/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[ 6364.9         4460.42       23011.30444444 ...  3966.48
 10713.55333333  1932.90888889]
Resultados salvos em results.csv


/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/clara/Documentos/enhanced-svd/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


,file,rate,method,rmse,nrmse,nrmse_mean,mae,mape,r2
0,232,10,Temporal_Matrix+SVD+KNN,370755.039226,0.125241,1.042760,254954.305541,175.887634,-0.104484
1,232,10,linear interpolation,327970.725624,0.110788,0.922427,218417.163599,105.314173,0.135718
2,232,10,knn imputer (k=5),353572.366416,0.119436,0.994433,256872.772183,223.056212,-0.004481
3,232,10,"kalman arima (1,1,1)",331586.875827,0.112010,0.932598,233484.254812,163.276167,0.116554
4,232,10,kalman structural (level),356257.644477,0.120344,1.001985,231784.819601,134.677525,-0.019797
